# 4. Merge Clinical and Corrected GEX_MAT

In [41]:
LIMMA_GENES = True

## Read datasets

In [42]:
import polars as pl

clinical = pl.read_csv(f"../dataset/created/clinical.csv")
clinical

,patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label,cobimetinib,dabrafenib,trametinib,vemurafenib,C195W,K601I,MND,V600E,V600K,V600R
i64,str,str,i64,str,str,str,f64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,"""BS_000""","""male""",56,"""IV""",null,"""normal""",30.5,"""no""","""no""","""no""","""Blateau et al.""",2,0,1,1,0,0,0,0,1,0,0
1,"""BS_001""","""male""",86,"""IV""",null,"""normal""",24.1,"""no""","""no""","""no""","""Blateau et al.""",2,0,1,0,0,0,0,0,1,0,0
2,"""BS_002""","""female""",47,"""IV""",null,"""normal""",14.1,"""no""","""no""","""no""","""Blateau et al.""",2,0,1,1,0,0,0,0,1,0,0
3,"""BS_003""","""female""",50,"""IV""",null,null,1.6,"""no""","""no""","""no""","""Blateau et al.""",0,0,0,0,1,0,0,0,1,0,0
4,"""BS_004""","""female""",47,"""IV""",null,"""elevated""",11.9,"""no""","""no""","""no""","""Blateau et al.""",1,0,1,1,0,0,0,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
381,"""HL_Shi-40""","""male""",47,"""IV""","""M1C""",null,3.0,"""no""","""no""","""no""","""Hugo et al.""",0,0,0,0,1,0,0,0,1,0,0
382,"""HL_Shi-41""","""male""",39,"""IV""","""M1A""",null,4.0,"""no""","""no""","""no""","""Hugo et al.""",0,0,0,0,1,0,0,0,1,0,0
383,"""HL_Shi-42""","""male""",84,"""IV""","""M1C""",null,8.0,"""no""","""no""","""no""","""Hugo et al.""",1,0,1,0,0,0,0,0,1,0,0


In [43]:
if LIMMA_GENES == True:
    gex_mat = pl.read_csv(f"../dataset/created/gex_mat_corrected_sig.csv")
    print(gex_mat)
else:
    gex_mat = pl.read_csv(f"../dataset/created/gex_mat_corrected.csv")
    # 1. Define which columns are the numeric data (samples)
    # We exclude 'HGNC' because it is a string/label
    sample_cols = [col for col in gex_mat.columns if col != 'HGNC']

    # 2. Compute variance for each row and add it as a new column
    # We use horizontal variance across the sample columns
    gex_with_var = gex_mat.with_columns(
        var = pl.concat_list(sample_cols).list.var()
    )

    # 3. Sort by variance descending and take the top 1000
    most_variable_genes = gex_with_var.sort("var", descending=True).head(1000)

    # 4. (Optional) Drop the 'var' column if you don't need it anymore
    gex_mat = most_variable_genes.drop("var")
    print(gex_mat)

shape: (157, 135)
┌──────────┬────────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ HGNC     ┆ 46225 PreC ┆ 19A      ┆ 28518_    ┆ … ┆ 05320372C ┆ 03660555B ┆ 04240120C ┆ 05320420B │
│ ---      ┆ ---        ┆ ---      ┆ 085G PreC ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str      ┆ f64        ┆ f64      ┆ ---       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
│          ┆            ┆          ┆ f64       ┆   ┆           ┆           ┆           ┆           │
╞══════════╪════════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ MAGEA4   ┆ 3.289116   ┆ 6.166987 ┆ 3.044772  ┆ … ┆ 4.275217  ┆ 4.298675  ┆ 4.275217  ┆ 4.303683  │
│ MAGEA1   ┆ 3.477783   ┆ 5.107743 ┆ 2.92707   ┆ … ┆ 4.464254  ┆ 5.143281  ┆ 4.466504  ┆ 4.542076  │
│ VCX3A    ┆ 3.493871   ┆ 5.419981 ┆ 4.06778   ┆ … ┆ 5.081262  ┆ 5.096021  ┆ 5.08705   ┆ 5.081262  │
│ SV2B     ┆ 5.186147   ┆ 4.823171 ┆ 3.939129  ┆ … ┆ 4.654132  ┆ 4.656211

In [44]:
sample_source = pl.read_csv(f"../dataset/created/sample_source.csv")
sample_source

sample_id,source,patientID,pfs_label
str,str,str,i64
"""46225 PreC""","""Long et al.""","""LR_SMU-020""",0
"""19A""","""Kwong et al.""","""KC_19""",2
"""28518_ 085G PreC""","""Long et al.""","""LR_MTP-034""",0
"""05320132B""","""Yan et al.""","""YR_2420""",0
"""05320425B""","""Yan et al.""","""YR_2425""",0
…,…,…,…
"""31721 PreC""","""Long et al.""","""LR_SMU-030""",0
"""05320372C""","""Yan et al.""","""YR_2337""",0
"""03660555B""","""Yan et al.""","""YR_3053""",0


## Convert GEX_MAT to patient-wised

In [45]:
gex_info = gex_mat.transpose(column_names='HGNC').insert_column(
    0, pl.Series('sample_id', gex_mat.columns[1:])
)
gex_info

sample_id,MAGEA4,MAGEA1,VCX3A,SV2B,BAIAP2L1,VCX2,CKMT1A,MAGEA10,CTAG2,GNB5,CSAG2,HES6,VCY,CTNNA3,POLN,CKMT1B,CHRAC1,SCRG1,COMMD9,XAGE1B,PTPMT1,CAMK2N1,XAGE1A,CD1D,LOC497256,PPP1R1C,LHX2,RBPMS2,DGCR5,FOXRED1,TMEM80,SEMA3G,IL1RAPL1,MAGEA12,MAGEA9B,SLITRK6,…,PBK,DENND5A,GAP43,HIGD1B,SCD5,TBC1D16,ABHD6,TCFL5,CRYZ,FXYD3,GRTP1,PANK1,NFE2L3,SNAP25,CSAG1,TRAF3,FAM53A,GDF7,KLF15,ZNF202,CDO1,PHACTR3,ADAMTSL1,VPS37D,CHAF1B,SLC25A25,IMMP1L,SLC6A20,MAGEA3,TNFAIP8L2,HERC2P4,PNMA2,SLC28A3,ALAD,RHCE,SORL1,SAMD4A
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""46225 PreC""",3.289116,3.477783,3.493871,5.186147,6.533368,4.289222,4.242716,2.744094,4.5036,5.581218,6.167682,6.819519,3.102561,4.130645,3.673891,3.830203,5.30408,5.461646,6.677661,5.704915,6.843471,5.652206,6.662992,5.575568,3.418506,3.764228,6.840774,6.110809,3.578879,5.74082,5.486852,3.210211,4.615067,6.616075,4.719096,3.362021,…,5.721356,6.936011,3.732348,3.988451,6.047171,7.315709,6.6019,6.020339,6.638424,5.102867,5.714226,5.954751,6.628995,3.314286,7.45174,4.760953,4.31633,3.767553,4.964254,3.680494,4.567317,3.216188,4.105782,5.373695,4.853604,6.021055,4.539725,4.258007,6.340998,5.387619,4.341656,5.212717,4.07715,6.453602,4.025915,5.836507,5.052871
"""19A""",6.166987,5.107743,5.419981,4.823171,5.311409,5.528453,4.778343,4.979167,4.364546,5.547944,4.987281,5.633712,5.133176,3.970293,4.613162,4.424071,5.241322,5.967647,6.972415,6.472569,6.59535,6.63216,7.394392,5.029068,4.188942,4.317182,5.310098,5.709997,4.177251,6.290476,5.527291,3.952704,4.882932,5.27739,4.560641,3.87326,…,5.485827,6.477572,4.367847,4.274909,4.80495,6.146125,5.393174,5.717921,6.126721,8.578424,4.575163,5.141814,6.114237,4.52205,6.548935,4.888543,4.892719,3.964284,4.831361,4.666712,4.829325,4.370325,4.552795,5.367545,5.34664,5.82663,4.358236,4.363163,5.665271,4.92579,4.342856,5.096919,3.961733,5.595386,4.051858,5.71987,5.414215
"""28518_ 085G PreC""",3.044772,2.92707,4.06778,3.939129,7.40999,4.019264,4.505902,1.834612,4.930342,6.35436,3.414913,6.871053,3.29299,4.199785,3.353117,4.252076,6.28516,7.110395,6.531652,3.239251,6.947778,6.489823,4.50535,4.973171,4.22036,3.41541,7.699444,7.232613,4.134796,5.405114,5.406499,3.880675,3.774336,2.934614,4.212075,3.327029,…,6.207326,7.424003,4.050523,5.476776,7.545229,7.904682,5.147701,7.346478,6.055302,5.125592,4.691147,5.635343,5.403858,5.989575,2.976295,5.24834,5.815276,3.601344,5.649623,4.611661,3.829223,4.907699,4.160146,6.161364,6.103216,6.836103,5.022999,4.661926,2.717848,4.647481,3.65326,7.611759,3.559913,6.822696,4.271709,6.192876,5.812424
"""05320132B""",4.282619,4.467322,5.081262,4.674186,5.785986,5.234033,4.998791,4.543894,4.630064,5.574422,4.70444,6.725942,4.825983,4.001732,4.45815,4.633107,5.55451,5.553009,6.479837,5.966891,6.684844,6.419655,6.829898,4.820773,4.114671,4.279911,5.666797,5.96874,4.308266,5.999008,5.392612,3.993457,4.400458,5.037674,4.477373,3.822252,…,5.867697,6.924071,4.328783,4.324114,5.11021,7.033699,5.774466,6.061279,6.425938,5.6751,4.750546,5.28657,6.413412,4.807269,5.834073,5.174292,5.041438,3.927749,5.076511,4.625109,4.995482,4.448656,4.440171,5.537879,5.611245,6.262184,4.700584,4.386395,5.147227,4.360259,4.192166,5.29168,3.947205,5.873933,4.048609,5.691446,5.164472
"""05320425B""",4.275824,4.468773,5.081262,4.674074,5.94996,5.234033,5.539528,4.541935,4.630064,5.514989,4.687101,6.720399,4.825983,4.016076,4.347127,5.31205,5.50128,5.498167,6.468583,5.994007,6.645701,6.400031,6.964703,4.797709,4.114671,4.195177,5.600778,6.016207,4.30886,5.89805,5.392406,4.123806,4.39059,4.902734,4.477373,3.883188,…,5.773432,6.498371,4.347056,4.353401,5.10951,7.062997,5.88375,6.114717,6.384005,5.911197,4.862692,5.339586,5.839447,4.813801,5.128129,4.652796,5.038255,3.930231,5.081848,4.585371,5.044745,4.47548,4.4688

In [46]:
gex = sample_source.join(other=gex_info, on='sample_id')
gex

sample_id,source,patientID,pfs_label,MAGEA4,MAGEA1,VCX3A,SV2B,BAIAP2L1,VCX2,CKMT1A,MAGEA10,CTAG2,GNB5,CSAG2,HES6,VCY,CTNNA3,POLN,CKMT1B,CHRAC1,SCRG1,COMMD9,XAGE1B,PTPMT1,CAMK2N1,XAGE1A,CD1D,LOC497256,PPP1R1C,LHX2,RBPMS2,DGCR5,FOXRED1,TMEM80,SEMA3G,IL1RAPL1,…,PBK,DENND5A,GAP43,HIGD1B,SCD5,TBC1D16,ABHD6,TCFL5,CRYZ,FXYD3,GRTP1,PANK1,NFE2L3,SNAP25,CSAG1,TRAF3,FAM53A,GDF7,KLF15,ZNF202,CDO1,PHACTR3,ADAMTSL1,VPS37D,CHAF1B,SLC25A25,IMMP1L,SLC6A20,MAGEA3,TNFAIP8L2,HERC2P4,PNMA2,SLC28A3,ALAD,RHCE,SORL1,SAMD4A
str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""46225 PreC""","""Long et al.""","""LR_SMU-020""",0,3.289116,3.477783,3.493871,5.186147,6.533368,4.289222,4.242716,2.744094,4.5036,5.581218,6.167682,6.819519,3.102561,4.130645,3.673891,3.830203,5.30408,5.461646,6.677661,5.704915,6.843471,5.652206,6.662992,5.575568,3.418506,3.764228,6.840774,6.110809,3.578879,5.74082,5.486852,3.210211,4.615067,…,5.721356,6.936011,3.732348,3.988451,6.047171,7.315709,6.6019,6.020339,6.638424,5.102867,5.714226,5.954751,6.628995,3.314286,7.45174,4.760953,4.31633,3.767553,4.964254,3.680494,4.567317,3.216188,4.105782,5.373695,4.853604,6.021055,4.539725,4.258007,6.340998,5.387619,4.341656,5.212717,4.07715,6.453602,4.025915,5.836507,5.052871
"""19A""","""Kwong et al.""","""KC_19""",2,6.166987,5.107743,5.419981,4.823171,5.311409,5.528453,4.778343,4.979167,4.364546,5.547944,4.987281,5.633712,5.133176,3.970293,4.613162,4.424071,5.241322,5.967647,6.972415,6.472569,6.59535,6.63216,7.394392,5.029068,4.188942,4.317182,5.310098,5.709997,4.177251,6.290476,5.527291,3.952704,4.882932,…,5.485827,6.477572,4.367847,4.274909,4.80495,6.146125,5.393174,5.717921,6.126721,8.578424,4.575163,5.141814,6.114237,4.52205,6.548935,4.888543,4.892719,3.964284,4.831361,4.666712,4.829325,4.370325,4.552795,5.367545,5.34664,5.82663,4.358236,4.363163,5.665271,4.92579,4.342856,5.096919,3.961733,5.595386,4.051858,5.71987,5.414215
"""28518_ 085G PreC""","""Long et al.""","""LR_MTP-034""",0,3.044772,2.92707,4.06778,3.939129,7.40999,4.019264,4.505902,1.834612,4.930342,6.35436,3.414913,6.871053,3.29299,4.199785,3.353117,4.252076,6.28516,7.110395,6.531652,3.239251,6.947778,6.489823,4.50535,4.973171,4.22036,3.41541,7.699444,7.232613,4.134796,5.405114,5.406499,3.880675,3.774336,…,6.207326,7.424003,4.050523,5.476776,7.545229,7.904682,5.147701,7.346478,6.055302,5.125592,4.691147,5.635343,5.403858,5.989575,2.976295,5.24834,5.815276,3.601344,5.649623,4.611661,3.829223,4.907699,4.160146,6.161364,6.103216,6.836103,5.022999,4.661926,2.717848,4.647481,3.65326,7.611759,3.559913,6.822696,4.271709,6.192876,5.812424
"""05320132B""","""Yan et al.""","""YR_2420""",0,4.282619,4.467322,5.081262,4.674186,5.785986,5.234033,4.998791,4.543894,4.630064,5.574422,4.70444,6.725942,4.825983,4.001732,4.45815,4.633107,5.55451,5.553009,6.479837,5.966891,6.684844,6.419655,6.829898,4.820773,4.114671,4.279911,5.666797,5.96874,4.308266,5.999008,5.392612,3.993457,4.400458,…,5.867697,6.924071,4.328783,4.324114,5.11021,7.033699,5.774466,6.061279,6.425938,5.6751,4.750546,5.28657,6.413412,4.807269,5.834073,5.174292,5.041438,3.927749,5.076511,4.625109,4.995482,4.448656,4.440171,5.537879,5.611245,6.262184,4.700584,4.386395,5.147227,4.360259,4.192166,5.29168,3.947205,5.873933,4.048609,5.691446,5.164472
"""05320425B""","""Yan et al.""","""YR_2425""",0,4.275824,4.468773,5.081262,4.674074,5.94996,5.234033,5.539528,4.541935,4.630064,5.514989,4.687101,6.720399,4.825983,4.016076,4.347127,5.31205,5.50128,5.498167,6.468583,5.994007,6.645701,6.400031,6.964703,4.797709,4.114671,4.195177,5.600778,6.016207,4.30886,5.89805,5.392406,4.123806,4.39059,…,5.773432,6.498371,4.347056,4.353401,5.10951,7.062997,5.88375,6.114717,6.384005,5.911197,4.862692,5.339586,5.839447,4.813801,5.128129,4.652796,5.038255,3.93023

## Merge with Clinical dataset

In [47]:
clinical_gex = clinical.join(gex, on='patientID', how='inner')[:, 1:]
clinical_gex

patientID,sex,age,AJCC_stage,M_stage,LDH,PFS_month,brain_metastasis,immunotherapy_treatment,pre_MAPKi_treatment,source,pfs_label,cobimetinib,dabrafenib,trametinib,vemurafenib,C195W,K601I,MND,V600E,V600K,V600R,sample_id,source_right,pfs_label_right,MAGEA4,MAGEA1,VCX3A,SV2B,BAIAP2L1,VCX2,CKMT1A,MAGEA10,CTAG2,GNB5,CSAG2,HES6,…,PBK,DENND5A,GAP43,HIGD1B,SCD5,TBC1D16,ABHD6,TCFL5,CRYZ,FXYD3,GRTP1,PANK1,NFE2L3,SNAP25,CSAG1,TRAF3,FAM53A,GDF7,KLF15,ZNF202,CDO1,PHACTR3,ADAMTSL1,VPS37D,CHAF1B,SLC25A25,IMMP1L,SLC6A20,MAGEA3,TNFAIP8L2,HERC2P4,PNMA2,SLC28A3,ALAD,RHCE,SORL1,SAMD4A
str,str,i64,str,str,str,f64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""YR_3053""","""male""",26,"""IV""","""M1B""","""elevated""",2.9,null,"""no""","""no""","""Yan et al.""",0,0,0,0,1,0,0,0,1,0,0,"""03660555B""","""Yan et al.""",0,4.298675,5.143281,5.096021,4.656211,5.647183,5.262724,4.878165,4.596075,4.635997,5.528943,4.788245,6.732841,…,5.903502,6.507592,4.388127,4.302565,5.036965,7.192134,5.777972,6.013873,6.382053,6.3116,4.682093,5.288554,6.566877,4.758247,7.009186,5.02649,5.033739,3.925451,5.095261,4.581815,4.902043,4.445664,4.279572,5.539303,5.788838,6.038979,4.377547,4.384223,6.284474,4.297704,4.262785,5.274553,3.888714,5.890571,4.040785,5.674289,5.297394
"""YR_3708""","""female""",58,"""IV""","""M1A""","""normal""",13.2,null,"""no""","""no""","""Yan et al.""",2,0,0,0,1,0,0,0,1,0,0,"""03660598B""","""Yan et al.""",2,6.516769,5.390489,5.117279,4.66819,5.775339,5.241646,4.863413,4.612415,4.630064,5.537698,4.974495,6.79111,…,6.083702,6.681839,4.319271,4.315734,5.074371,7.393495,6.047259,6.161249,7.085179,7.588654,4.688986,5.334356,5.758516,4.873319,7.236846,4.815795,5.054274,3.922493,5.132568,4.549927,4.893729,4.4546,4.316156,5.543453,5.774022,6.567069,4.577433,4.385748,6.862708,4.276909,4.483725,5.299898,3.846581,6.463356,4.002404,5.224376,5.490748
"""YR_3705""","""male""",45,"""IV""","""M1A""","""normal""",17.4,null,"""no""","""no""","""Yan et al.""",2,0,0,0,1,0,0,0,1,0,0,"""03660602B""","""Yan et al.""",2,5.067833,4.897864,5.565421,4.65066,5.590498,5.246197,4.862598,4.543948,4.630064,5.665834,4.745568,6.715236,…,5.907191,6.655564,4.36465,4.30424,5.107941,7.029765,5.764098,6.049477,6.686661,6.379655,4.717485,5.258418,6.591138,4.768234,6.107981,5.271509,5.035231,3.922493,5.074602,4.610968,4.892663,4.449559,4.333312,5.547173,5.611036,6.097599,4.739828,4.384223,5.626812,4.337528,4.258745,5.309691,3.851291,5.914027,4.059241,6.147752,5.007887
"""YR_1106""","""male""",48,"""IIIC""",null,"""normal""",1.4,null,"""no""","""no""","""Yan et al.""",0,1,0,0,1,0,0,0,1,0,0,"""04240076F""","""Yan et al.""",0,4.276826,4.498819,5.081262,4.677048,5.721581,5.234033,4.864543,4.539119,4.640348,5.534643,4.683303,6.731672,…,5.753568,7.031527,4.319476,4.321546,5.079875,7.051977,5.770349,6.00381,6.490751,5.489067,4.696509,5.263581,5.740149,4.830475,5.1275,4.915295,5.041552,3.922493,5.069316,4.592879,4.942075,4.447395,4.652545,5.538047,5.448097,6.070584,4.415937,4.385166,4.948698,4.413269,4.219724,5.321396,3.871215,5.981546,4.049749,5.33609,5.050745
"""YR_1110""","""male""",33,"""IV""","""M1C""","""normal""",1.4,null,"""no""","""no""","""Yan et al.""",0,1,0,0,1,0,0,0,1,0,0,"""04240106F""","""Yan et al.""",0,4.27679,4.820164,5.092965,4.716673,5.731831,5.238127,4.861666,4.827292,4.630064,5.393001,4.835158,6.758138,…,5.663398,7.146226,4.309754,4.318632,5.028025,7.254626,6.009336,6.29628,6.625682,5.548108,4.757115,5.373615,6.141256,4.766991,6.675747,4.383612,5.042648,3.924168,5.153256,4.66547,4.905537,4.47972,4.350591,5.534836,5.658816,6.142435,4.519256,4.385145,6.264354,4.295142,4.685055,5.30119,3.859265,6.369937,4.339283,6.487131,5.317965
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [48]:
clinical_gex.write_csv(f"../dataset/created/clinical_gex.csv")